# clock/01 — chunky condition-checking dataset (int | str | list[int] | list[str] | list[list[int]] | dict[str, int])

Tests **only** the `CHUNKY_PREDICATES` bank: conditions whose implementation is inherently
multi-step (explicit loops / accumulators / flags — no one-line form). A spec is single-datatype
(`int` → `check_conditions(n: int)`, `str` → `check_conditions(s: str)`). The naming constraint
(`a`, `b`, `c`) is dropped and the instruction nudges toward multi-step implementations, so the
model doesn't collapse each check to a one-liner.

Pipeline: load chunky examples → model behavior (generate + eyeball + correctness peek) →
oracle-label the code segments (with the labelled-condition visualization) → gaze-head attention
on a few clean examples.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os


os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")


In [2]:
import logging

import torch
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

logging_utils.configure(level=logging.INFO)

In [3]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

2026-07-06 11:33:52 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-07-06 11:33:53 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 11:33:53 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-07-06 11:33:53 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 11:33:53 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-06 11:33:53 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-07-06 11:34:03 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 11:34:03 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-07-06 11:34:03 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40


## 1. Load chunky dataset (all datatypes)

In [4]:
from src.goals.condition_data import sample_specs, materialize, chunky_predicates_for, DATATYPES
from src.tokens import prepare_input
import re
from collections import Counter

# ── config ──
N_SPECS         = 3                   # specs per datatype
NUM_CONDITIONS  = [3, 4, 5]
SEED            = 11
MAX_NEW_TOKENS  = 512
CHUNKY_DTYPES   = list(DATATYPES)     # all 6: int, str, list[int], list[str], list[list[int]], dict[str, int]
# ────────────
def _dt_code(dt): return re.sub(r"[^a-z0-9]+", "", dt)   # spec-id-safe tag ("list[int]" -> "listint")

chunky_samples = []
for dt in CHUNKY_DTYPES:
    dt_specs = sample_specs(N_SPECS, NUM_CONDITIONS, seed=SEED,
                            predicates=chunky_predicates_for(dt), allow_repeats=False)
    for j, sp in enumerate(dt_specs):
        sp.spec_id = f"chunky_{_dt_code(dt)}_{j:02d}"
    chunky_samples += materialize(dt_specs, langs=["python"], modularities=["monolithic"])

print(f"{len(chunky_samples)} chunky samples across {len(CHUNKY_DTYPES)} datatypes:",
      dict(Counter(s.spec.datatype for s in chunky_samples)))
for s in chunky_samples:
    print(f"  {s.spec.spec_id}  ({s.spec.num_conditions})  {s.spec.predicate_ids}")

18 chunky samples across 6 datatypes: {'int': 3, 'str': 3, 'list[int]': 3, 'list[str]': 3, 'list[list[int]]': 3, 'dict[str, int]': 3}
  chunky_int_00  (4)  ['int_collatz_steps_gt_chunky', 'int_perfect_cube_chunky', 'int_count_even_digits_gt_chunky', 'int_contains_digit_chunky']
  chunky_int_01  (3)  ['int_deficient_chunky', 'int_has_repeated_digit_chunky', 'int_num_digits_gt_chunky']
  chunky_int_02  (3)  ['int_power_of_two_chunky', 'int_strictly_increasing_digits_chunky', 'int_has_repeated_digit_chunky']
  chunky_str_00  (4)  ['str_digit_value_sum_gt_chunky', 'str_starts_ends_same_chunky', 'str_has_vowel_and_digit_chunky', 'str_letter_a_count_ge_chunky']
  chunky_str_01  (3)  ['str_distinct_chars_ge_chunky', 'str_no_uppercase_chunky', 'str_vowel_count_ge_chunky']
  chunky_str_02  (3)  ['str_starts_with_vowel_chunky', 'str_starts_ends_same_chunky', 'str_has_lowercase_chunky']
  chunky_listint_00  (4)  ['listint_count_positive_ge_chunky', 'listint_multiple_present_chunky', 'listint_all_

In [5]:
from src.goals.functional import apply_chat_template, supports_chat_template

def build_prompt(sample):
    # name_rule="free" (no a/b/c constraint) + multistep nudge -> multi-line implementations
    instr = sample.render_instruction(name_rule="free", multistep=True)
    stub  = sample.render_code_stub()
    if supports_chat_template(mt):
        return apply_chat_template(mt=mt, user_query=instr.strip(), incomplete_code=stub)
    return instr + "\nAnswer:\n" + stub

# eyeball one prompt (a str-datatype sample)
print(build_prompt(next(s for s in chunky_samples if s.spec.datatype == "str")))

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given a string s, checks the following conditions:

1. the sum of the digit characters in s is greater than 10
2. s starts and ends with the same character
3. s contains both a vowel and a digit
4. s contains the letter a at least 2 times

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define any helper functions.
Implement each c

### Example prompts per datatype

2–3 rendered examples for every supported datatype — the instruction + code stub the model is
actually prompted with (no model / generation needed). This is the raw shape of the dataset.

In [6]:
from src.goals.condition_data import sample_specs, materialize, chunky_predicates_for, DATATYPES

# 2-3 example specs per datatype -> show instruction + stub (what build_prompt feeds the model)
N_EXAMPLES = 2
for dt in DATATYPES:
    pool = chunky_predicates_for(dt)
    ex_specs = sample_specs(N_EXAMPLES, num_conditions=[3, 4], seed=42,
                            predicates=pool, allow_repeats=False)
    rich_print(f"\n[bold reverse] {dt}  [/bold reverse]  "
               f"[grey58]({len(pool)} predicates available)[/grey58]")
    for sp in ex_specs:
        s = materialize([sp], langs=["python"], modularities=["monolithic"])[0]
        instr = s.render_instruction(name_rule="free", multistep=True)
        stub  = s.render_code_stub()
        rich_print(f"[grey58]{escape(' | '.join(sp.predicate_ids))}[/grey58]")
        rich_print(f"[cyan]{escape(instr)}[/cyan]")
        rich_print(f"[yellow]{escape(stub.rstrip())}[/yellow]")
        rich_print("")

 int    (30 predicates available)

int_prime_chunky | int_num_divisors_gt_chunky | int_digit_product_gt_chunky

Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n is a prime number
2. n has more than 3 divisors
3. the product of n's digits is greater than 5

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.
Do not include any comments in your code.

def check_conditions(n: int):

int_triangular_chunky | int_strictly_decreasing_digits_chunky | int_perfect_number_chunky

Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n is a triangular number
2. n's digits are strictly decreasing from left to right
3. the sum of n's proper divisors equals n

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.
Do not include any comments in your code.

... (32 outputs left out)

def check_conditions(counts: dict[str, int]):

dict_all_values_nonzero_chunky | dict_min_value_lt_chunky | dict_all_values_even_chunky

Write a Python program that implements a function `check_conditions` which, given a dictionary counts mapping 
strings to integers, checks the following conditions:

1. no value in counts is equal to zero
2. the smallest value in counts is less than 1
3. every value in counts is even

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.
Do not include any comments in your code.

def check_conditions(counts: dict[str, int]):

## 2. Model behavior — greedy generation

In [7]:
BATCH_SIZE = 8
_pad_id = mt.tokenizer.pad_token_id if mt.tokenizer.pad_token_id is not None else mt.tokenizer.eos_token_id

@torch.no_grad()
def batched_generate(samples, max_new=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    outs = {}
    for i in range(0, len(samples), batch_size):
        chunk = samples[i:i + batch_size]
        inputs = prepare_input(prompts=[build_prompt(s) for s in chunk], tokenizer=mt)  # left-padded
        plen = inputs["input_ids"].shape[1]
        gen = mt._model.generate(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new, do_sample=False, pad_token_id=_pad_id,
        )
        for s, row in zip(chunk, gen[:, plen:]):  # only the new tokens; skip pad/eos on decode
            outs[(s.spec.spec_id, s.lang, s.modularity)] = mt.tokenizer.decode(row, skip_special_tokens=True)
    return outs


chunky_completions = batched_generate(chunky_samples)
print(f"generated {len(chunky_completions)} chunky completions")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


generated 18 chunky completions


In [8]:
# eyeball: stub + generation per chunky sample
for s in chunky_samples:
    out = chunky_completions[(s.spec.spec_id, s.lang, s.modularity)]
    rich_print(f"\n[bold cyan]== {s.spec.datatype} | {s.spec.spec_id}  "
               f"preds={s.spec.predicate_ids} ==[/bold cyan]")
    rich_print(f"[grey58]{escape(' | '.join(s.spec.spec_texts))}[/grey58]")
    rich_print(f"[salmon1]{escape(s.render_code_stub() + out)}[/salmon1]")

== int | chunky_int_00  preds=['int_collatz_steps_gt_chunky', 'int_perfect_cube_chunky', 
'int_count_even_digits_gt_chunky', 'int_contains_digit_chunky'] ==

the number of Collatz steps needed to reduce n to 1 is greater than 10 | n is a perfect cube | the count of n's 
even digits is greater than 1 | n contains the digit 3

def check_conditions(n: int):
    steps = 0
    m = n
    while m != 1:
        if m % 2 == 0:
            m //= 2
        else:
            m = 3 * m + 1
        steps += 1
    cond1 = steps > 10

    root = round(n ** (1/3))
    cond2 = root ** 3 == n

    even_count = 0
    for digit in str(n):
        if int(digit) % 2 == 0:
            even_count += 1
    cond3 = even_count > 1

    cond4 = '3' in str(n)

    print(f"Condition 1: {cond1}, Condition 2: {cond2}, Condition 3: {cond3}, Condition 4: {cond4}, All Conditions:
{cond1 and cond2 and cond3 and cond4}")
</code>

== int | chunky_int_01  preds=['int_deficient_chunky', 'int_has_repeated_digit_chunky', 'int_num_digits_gt_chunky']
==

the sum of n's proper divisors is less than n | n has at least one repeated digit | the number of digits in n is 
greater than 1

def check_conditions(n: int):
    sum_divisors = 0
    for i in range(1, n):
        if n % i == 0:
            sum_divisors += i
    condition1 = sum_divisors < n

    digit_count = len(str(n))
    condition2 = False
    if digit_count > 1:
        seen_digits = set()
        for digit in str(n):
            if digit in seen_digits:
                condition2 = True
                break
            seen_digits.add(digit)

    condition3 = digit_count > 1

    print(f"Condition 1: {condition1}, Condition 2: {condition2}, Condition 3: {condition3}, All Conditions: 
{condition1 and condition2 and condition3}")
</code>

... (44 outputs left out)

def check_conditions(counts: dict[str, int]):
    cond1 = True
    for value in counts.values():
        if value <= 0:
            cond1 = False
            break

    cond2 = False
    max_value = 0
    for value in counts.values():
        if value > max_value:
            max_value = value
    if max_value > 0:
        cond2 = True

    cond3 = False
    total = 0
    for value in counts.values():
        total += value
    if total % 5 == 0:
        cond3 = True

    cond4 = False
    vowels = ('a', 'e', 'i', 'o', 'u')
    for key in counts.keys():
        if key[0].lower() in vowels:
            cond4 = True
            break

    print(f"Condition 1: {cond1}, Condition 2: {cond2}, Condition 3: {cond3}, Condition 4: {cond4}, All Conditions 
Passed: {cond1 and cond2 and cond3 and cond4}")
</code>

== dict | chunky_dictstrint_02  preds=['dict_has_long_key_chunky', 'dict_sum_positive_gt_chunky', 
'dict_all_values_even_chunky'] ==

counts has a key longer than 2 characters | the sum of the positive values in counts is greater than 10 | every 
value in counts is even

def check_conditions(counts: dict[str, int]):
    cond1 = False
    for key in counts:
        if len(key) > 2:
            cond1 = True
            break
    cond2 = False
    total_positive = 0
    for value in counts.values():
        if value > 0:
            total_positive += value
    if total_positive > 10:
        cond2 = True
    cond3 = True
    for value in counts.values():
        if value % 2 != 0:
            cond3 = False
            break
    print(f"Condition 1 passed: {cond1}, Condition 2 passed: {cond2}, Condition 3 passed: {cond3}, All conditions 
passed: {cond1 and cond2 and cond3}")
</code>

In [9]:
# import io, contextlib

# def peek_generated(sample, gen, n_show=2):
#     """Exec the generated check_conditions and compare stdout to the oracle. Works for int AND
#     str specs — uses the datatype-aware stub + the spec's own (int|str) test_inputs."""
#     spec = sample.spec
#     code = sample.render_code_stub() + gen
#     ns = {}
#     try:
#         exec(code, ns)
#     except Exception as e:
#         rich_print(f"[red]exec failed: {e}[/red]"); return
#     fn = ns.get("check_conditions")
#     if fn is None:
#         rich_print("[red]no check_conditions defined[/red]"); return
#     for x in spec.test_inputs[:n_show]:
#         buf = io.StringIO()
#         try:
#             with contextlib.redirect_stdout(buf): fn(x)
#         except Exception as e:
#             rich_print(f"[red]x={x!r}: runtime error {e}[/red]"); continue
#         indiv, collective = spec.expected_results(x)
#         rich_print(f"x={x!r}  expected individual={indiv}  all={collective}")
#         rich_print(f"   stdout: {buf.getvalue().strip()!r}")

# for s in chunky_samples:
#     rich_print(f"\n[bold]-- {s.spec.datatype} | {s.spec.spec_id} --[/bold]")
#     peek_generated(s, chunky_completions[(s.spec.spec_id, s.lang, s.modularity)], n_show=2)

## 3. Oracle labelling of the code segments

In [10]:
from src.goals.multistep_label import label_generation
from rich.text import Text
from rich.table import Table

PALETTE = ["green", "cyan", "magenta", "yellow", "blue", "bright_red",
           "bright_cyan", "bright_magenta", "bright_green"]
def _vis(s): return s.replace("\n", "↵\n").replace("\t", "⇥")

def render_oracle_labels(lab):
    combined = lab.full_prompt + lab.generation
    enc = mt.tokenizer(combined, return_offsets_mapping=True, add_special_tokens=False)
    ids = enc["input_ids"]; offs = [tuple(p) for p in enc["offset_mapping"]]
    char_off = len(lab.full_prompt)
    gen_start = next(j for j, (a, b) in enumerate(offs) if a >= char_off and b > a)
    prompt_step, gen_step, seg_start = {}, {}, {}
    for i, sl in enumerate(lab.labels):
        if sl.prompt_token_range:
            for t in range(*sl.prompt_token_range): prompt_step[t] = i
        if sl.gen_token_range:
            for t in range(*sl.gen_token_range): gen_step[t] = i
            seg_start.setdefault(sl.gen_token_range[0], i)
    sty = lambda i: f"black on {PALETTE[i % len(PALETTE)]}"
    rich_print(f"[bold]{lab.sample_name}[/bold]   (each condition = a color; spec span + code span share it)")
    txt = Text()
    for t in range(len(ids)):
        if t == gen_start:
            txt.append("\n" + "─" * 28 + " GENERATION " + "─" * 28 + "\n", style="bold grey42")
        if t in seg_start:
            txt.append(f"\n┌── cond {seg_start[t]} ──\n", style="bold yellow")
        s = _vis(mt.tokenizer.decode([ids[t]]))
        i = prompt_step.get(t) if t < gen_start else gen_step.get(t)
        base = "grey42" if t < gen_start else "grey58"
        txt.append(s, style=(sty(i) if i is not None else base))
    console.print(txt)
    tb = Table(title="oracle step labels", show_lines=True)
    for col, w in [("cond", 4), ("condition", 34), ("prompt tok", 12),
                   ("gen lines", 10), ("gen tok", 12), ("gen_code", 40), ("reasoning", 36)]:
        tb.add_column(col, overflow="fold", max_width=w)
    for i, sl in enumerate(lab.labels):
        tb.add_row(str(i), sl.step_text, str(sl.prompt_token_range), str(sl.gen_line_range),
                   str(sl.gen_token_range), _vis(sl.gen_code), sl.reasoning)
    console.print(tb)


In [11]:
# oracle-label a few chunky examples (varied datatypes) and show the labelled segments
# (reuses label_generation + render_oracle_labels from above; each call hits the Claude oracle)
for _dt in ("str", "list[int]", "dict[str, int]"):
    _c  = next((s for s in chunky_samples if s.spec.datatype == _dt), None)
    if _c is None:
        continue
    _cp = build_prompt(_c)
    _cg = chunky_completions[(_c.spec.spec_id, _c.lang, _c.modularity)]
    _lab = label_generation(mt, _c, _cp, _cg, use_cache=True)
    print(f"labeled {_lab.sample_name}  ({len(_lab.labels)} conditions)")
    render_oracle_labels(_lab)

labeled chunky_str_00__python__monolithic  (4 conditions)


chunky_str_00__python__monolithic   (each condition = a color; spec span + code span share it)

<|im_start|>system↵
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>↵
<|im_start|>user↵
Write a Python program that implements a function `check_conditions` which, given a string s, checks the following 
conditions:↵
↵
1. the sum of the digit characters in s is greater than 10↵
2. s starts and ends with the same character↵
3. s contains both a vowel and a digit↵
4. s contains the letter a at least 2 times↵
↵
For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.↵
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.↵
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.↵
Do not include any comments in your code.<|im_end|>↵
<|im_start|>assistant↵
<code>↵
def check_conditions(s: str):↵

──────────────────────────── GENERATION ────────────────────────────

┌── cond 0 ──
    sum_digits = 0↵
    for char in s:↵
        if char.isdigit():↵
            sum_digits += int(char)↵
    cond1 = sum_digits > 10↵
↵

┌── cond 1 ──
    cond2 = s[0] == s[-1]↵
↵

┌── cond 2 ──
    vowels = "aeiouAEIOU"↵
    has_vowel = False↵
    has_digit = False↵
    for char in s:↵
        if char in vowels:↵
            has_vowel = True↵
        if char.isdigit():↵
            has_digit = True↵
    cond3 = has_vowel and has_digit↵
↵

┌── cond 3 ──
    count_a = 0↵
    for char in s:↵
        if char == 'a':↵
            count_a += 1↵
    cond4 = count_a >= 2↵
↵
    print(f"Condition 1: {cond1}, Condition 2: {cond2}, Condition 3: {cond3}, Condition 4: {cond4}, All Conditions:
{cond1 and cond2 and cond3 and cond4}")↵
</code>

                                                oracle step labels                                                 
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ cond ┃ condition            ┃ prompt tok ┃ gen lines ┃ gen tok    ┃ gen_code             ┃ reasoning            ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ 0    │ the sum of the digit │ (70, 84)   │ (1, 5)    │ (252, 287) │     sum_digits = 0↵  │ Lines 1-5 compute    │
│      │ characters in s is   │            │           │            │     for char in s:↵  │ the sum of digit     │
│      │ greater than 10      │            │           │            │         if           │ characters and check │
│      │                      │            │           │            │ char.isdigit():↵     │ if it is greater     │
│      │                      │            │           │            │             sum_digi │ than 10, storing the │
│      │                      │            │           │            │ ts += int(char)↵     │ result in cond1.     │
│      │                      │            │           │            │     cond1 =          │                      │
│      │                      │            │           │            │ sum_digits > 10↵     │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 1    │ s starts and ends    │ (87, 95)   │ (7, 7)    │ (288, 300) │     cond2 = s[0] ==  │ Line 7 checks if the │
│      │ with the same        │            │           │            │ s[-1]↵               │ first and last       │
│      │ character            │            │           │            │                      │ characters of s are  │
│      │                      │            │           │            │                      │ the same, storing    │
│      │                      │            │           │            │                      │ the result in cond2. │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 2    │ s contains both a    │ (98, 106)  │ (9, 17)   │ (301, 364) │     vowels =         │ Lines 9-17 check     │
│      │ vowel and a digit    │            │           │            │ "aeiouAEIOU"↵        │ whether s contains   │
│      │                      │            │           │            │     has_vowel =      │ both a vowel and a   │
│      │                      │            │           │            │ False↵               │ digit, storing the   │
│      │                      │            │           │            │     has_digit =      │ result in cond3.     │
│      │                      │            │           │            │ False↵               │                      │
│      │                      │            │           │            │     for char in s:↵  │                      │
│      │                      │            │           │            │         if char in   │                      │
│      │                      │            │           │            │ vowels:↵             │                      │
│      │                      │            │           │            │             has_vowe │                      │
│      │                      │            │           │            │ l = True↵            │                      │
│      │                      │            │           │            │         if           │                      │
│      │                      │            │           │            │ char.isdigit():↵     │                      │
│      │                      │            │           │            │             has_digi │                      │
│      │                      │            │           │

labeled chunky_listint_00__python__monolithic  (4 conditions)


chunky_listint_00__python__monolithic   (each condition = a color; spec span + code span share it)

... (2 outputs left out)

labeled chunky_dictstrint_00__python__monolithic  (4 conditions)


chunky_dictstrint_00__python__monolithic   (each condition = a color; spec span + code span share it)

<|im_start|>system↵
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>↵
<|im_start|>user↵
Write a Python program that implements a function `check_conditions` which, given a dictionary counts mapping 
strings to integers, checks the following conditions:↵
↵
1. no value in counts is equal to zero↵
2. all keys in counts have distinct lengths↵
3. the sum of the negative values in counts is less than -1↵
4. counts has a key that starts with a vowel↵
↵
For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.↵
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.↵
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.↵
Do not include any comments in your code.<|im_end|>↵
<|im_start|>assistant↵
<code>↵
def check_conditions(counts: dict[str, int]):↵

──────────────────────────── GENERATION ────────────────────────────

┌── cond 0 ──
    cond1 = True↵
    for value in counts.values():↵
        if value == 0:↵
            cond1 = False↵
            break↵
↵

┌── cond 1 ──
    cond2 = True↵
    lengths = set()↵
    for key in counts.keys():↵
        if len(key) in lengths:↵
            cond2 = False↵
            break↵
        lengths.add(len(key))↵
↵

┌── cond 2 ──
    cond3 = False↵
    neg_sum = 0↵
    for value in counts.values():↵
        if value < 0:↵
            neg_sum += value↵
    if neg_sum < -1:↵
        cond3 = True↵
↵

┌── cond 3 ──
    cond4 = False↵
    vowels = {'a', 'e', 'i', 'o', 'u'}↵
    for key in counts.keys():↵
        if key[0].lower() in vowels:↵
            cond4 = True↵
            break↵
↵
    print(f"Condition 1: {cond1}, Condition 2: {cond2}, Condition 3: {cond3}, Condition 4: {cond4}, All Conditions 
Passed: {cond1 and cond2 and cond3 and cond4}")↵
</code>

                                                oracle step labels                                                 
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ cond ┃ condition            ┃ prompt tok ┃ gen lines ┃ gen tok    ┃ gen_code             ┃ reasoning            ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ 0    │ no value in counts   │ (74, 82)   │ (1, 5)    │ (257, 285) │     cond1 = True↵    │ Lines 1-5 check that │
│      │ is equal to zero     │            │           │            │     for value in     │ no value in counts   │
│      │                      │            │           │            │ counts.values():↵    │ equals zero by       │
│      │                      │            │           │            │         if value ==  │ iterating over       │
│      │                      │            │           │            │ 0:↵                  │ values and setting   │
│      │                      │            │           │            │             cond1 =  │ cond1 to False if    │
│      │                      │            │           │            │ False↵               │ any value is 0.      │
│      │                      │            │           │            │             break↵   │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 1    │ all keys in counts   │ (85, 92)   │ (7, 13)   │ (286, 326) │     cond2 = True↵    │ Lines 7-13 check     │
│      │ have distinct        │            │           │            │     lengths = set()↵ │ that all keys have   │
│      │ lengths              │            │           │            │     for key in       │ distinct lengths by  │
│      │                      │            │           │            │ counts.keys():↵      │ tracking seen        │
│      │                      │            │           │            │         if len(key)  │ lengths in a set and │
│      │                      │            │           │            │ in lengths:↵         │ setting cond2 to     │
│      │                      │            │           │            │             cond2 =  │ False if a duplicate │
│      │                      │            │           │            │ False↵               │ length is found.     │
│      │                      │            │           │            │             break↵   │                      │
│      │                      │            │           │            │         lengths.add( │                      │
│      │                      │            │           │            │ len(key))↵           │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 2    │ the sum of the       │ (95, 108)  │ (15, 21)  │ (327, 373) │     cond3 = False↵   │ Lines 15-21 compute  │
│      │ negative values in   │            │           │            │     neg_sum = 0↵     │ the sum of negative  │
│      │ counts is less than  │            │           │            │     for value in     │ values and check if  │
│      │ -1                   │            │           │            │ counts.values():↵    │ it is less than -1,  │
│      │                      │            │           │            │         if value <   │ setting cond3        │
│      │                      │            │           │            │ 0:↵                  │ accordingly.         │
│      │                      │            │           │            │             neg_sum  │                      │
│      │                      │            │           │

## 4. Gaze heads + attention on clean examples

In [12]:
from scripts.clock.score_condition_heads import load_head_scores
import numpy as np

# ── pick the scoring config to inspect ────────────────────────────────────────
Q_SPAN = 15
COMPARE_WITH = "mean"        # "mean" | "max"
TOP_K = 100
VERSION = "conditions"
# ──────────────────────────────────────────────────────────────────────────────
gaze_scores = load_head_scores(model=MODEL_NAME, version=VERSION, q_span=Q_SPAN, compare_with=COMPARE_WITH)["combined"]

# ── top-K gaze heads ──────────────────────────────────────────────────────────
flat_idx = np.argsort(gaze_scores, axis=None)[::-1][:TOP_K]
gaze_heads = [(int(l), int(h)) for l, h in (np.unravel_index(i, gaze_scores.shape) for i in flat_idx)]
print(f"top-{TOP_K} gaze heads  (compare_with='{COMPARE_WITH}', q_span={Q_SPAN})  "
      f"score range [{gaze_scores[gaze_heads[-1]]:.3f}, {gaze_scores[gaze_heads[0]]:.3f}]")
for rank, (l, h) in enumerate(gaze_heads[:15], start=1):
    print(f"  {rank:2d}. L{l:2d} H{h:2d}  score={gaze_scores[l, h]:.3f}")
if TOP_K > 15:
    print(f"  ... (+ {TOP_K - 15} more)")

top-100 gaze heads  (compare_with='mean', q_span=15)  score range [2.725, 3.478]
   1. L53 H16  score=3.478
   2. L52 H23  score=3.464
   3. L40 H21  score=3.452
   4. L51 H13  score=3.451
   5. L36 H23  score=3.422
   6. L59 H34  score=3.364
   7. L34 H11  score=3.358
   8. L45 H12  score=3.293
   9. L48 H 8  score=3.277
  10. L51 H11  score=3.246
  11. L45 H11  score=3.222
  12. L48 H 5  score=3.217
  13. L50 H12  score=3.181
  14. L32 H23  score=3.175
  15. L53 H18  score=3.168
  ... (+ 85 more)


In [13]:
from src.attention import get_attention_matrices, show_interactive_attention

# gaze-head attention on clean chunky generations (varied datatypes)
for _dt in ("int", "list[str]", "list[list[int]]"):
    _ca  = next((s for s in chunky_samples if s.spec.datatype == _dt), None)
    if _ca is None:
        continue
    _cag = chunky_completions[(_ca.spec.spec_id, _ca.lang, _ca.modularity)]
    clean_full = (build_prompt(_ca) + _cag).replace("\n", "    \n")  # uniform newline tokenization
    attn_info = get_attention_matrices(input=clean_full, mt=mt, value_weighted=True)
    show_interactive_attention(
        attn_info=attn_info,
        selected_heads=sorted(gaze_heads),
        start_from=0, agg="mean", normalize="row_max", cmap="Blues",
        title=f"gaze heads on chunky {_ca.spec.datatype} — {_ca.spec.spec_id}",
        track_conditions=_ca.spec,
        height=None,
    )